In [3]:
import numpy as np

# 第 4 章：单权重学习

这一节先把问题压缩到最小：只有一个输入、一个权重和一个目标值。目标不是马上记住求导公式，而是看清楚一个训练步究竟做了什么。

```text
预测 → 计算误差 → 判断更新方向 → 更新权重
```

前向计算：`prediction = x * weight`。

误差使用平方误差：`error = (prediction - target) ** 2`。平方可以保证误差非负，并且让较大的错误受到更强惩罚。

`delta = prediction - target` 表示当前预测偏离目标的方向；`weight_delta = delta * x` 把这个方向传回权重；最后用学习率 `alpha` 控制每次移动的步长。

当 `prediction < target` 时，`delta` 为负，更新式 `weight = weight - alpha * weight_delta` 会让权重增加；当 `prediction > target` 时，权重会往相反方向移动。

严格求平方误差的导数会多出一个因子 `2`。本实验省略这个常数，把它视为学习率的一部分，因此重点放在更新方向和形状，而不是常数倍。

本例中目标关系是 `2 * weight = 1`，所以理想权重是 `0.5`。运行输出时重点观察：权重逐渐接近 `0.5`，预测逐渐接近 `1.0`，误差逐渐接近 `0`。注意代码打印的 `weight` 已经更新，但 `prediction` 是更新前计算的值；如果要让日志完全对应，应在更新后重新计算预测。

In [1]:
x = 2
weight = 0.3
target = 1.0
alpha = 0.1

for step in range(20):
    prediction = x * weight
    error = (prediction - target) ** 2

    delta = prediction - target
    weight_delta = delta * x
    weight = weight - weight_delta * alpha

    new_prediction = x * weight
    new_error = (new_prediction - target) ** 2

    print(
        step,
        "weight=", weight,
        "prediction=", prediction,
        "error=", new_error
    )

0 weight= 0.38 prediction= 0.6 error= 0.0576
1 weight= 0.428 prediction= 0.76 error= 0.020736000000000004
2 weight= 0.4568 prediction= 0.856 error= 0.007464960000000006
3 weight= 0.47408 prediction= 0.9136 error= 0.0026873856
4 weight= 0.484448 prediction= 0.94816 error= 0.0009674588160000013
5 weight= 0.4906688 prediction= 0.968896 error= 0.0003482851737599988
6 weight= 0.49440128 prediction= 0.9813376 error= 0.00012538266255360006
7 weight= 0.496640768 prediction= 0.98880256 error= 4.5137758519295725e-05
8 weight= 0.49798446080000003 prediction= 0.993281536 error= 1.6249593066946105e-05
9 weight= 0.49879067648000003 prediction= 0.9959689216000001 error= 5.84985350410049e-06
10 weight= 0.49927440588800004 prediction= 0.9975813529600001 error= 2.1059472614760476e-06
11 weight= 0.4995646435328 prediction= 0.9985488117760001 error= 7.581410141314544e-07
12 weight= 0.49973878611968 prediction= 0.9991292870656 error= 2.7293076508737e-07
13 weight= 0.499843271671808 prediction= 0.9994775722

# 第 5 章：一个样本、多个权重

现在一个样本包含 3 个输入：`[2, 1, -1]`，对应 3 个权重：`[w1, w2, w3]`。这一节要建立的直觉是：一个误差如何同时影响多个权重。

预测公式从标量乘法变成点积：

```python
prediction = np.dot(inputs, weights)
```

形状关系是：

```text
inputs  (3,)
weights (3,)
          ↓ 点积
prediction：标量
```

一个标量误差要同时反馈给 3 个权重，因此更新量写成 `weight_deltas = delta * inputs`。这里 NumPy 会把标量 `delta` 广播到输入向量的每个位置。第 `i` 个权重的梯度都包含对应的输入 `inputs[i]`：输入绝对值越大，该权重受到的更新通常越大。

注意：只有一个样本时，通常不能唯一确定所有权重。训练只保证加权总和正确，不保证恢复某一组唯一的权重。可以把它看成一个方程约束多个未知数：模型可能找到很多组都能产生同一个预测的权重。

In [ ]:
inputs = np.array([2., 1., -1.])
weights = np.array([0.5, 0.2, 0.1])
target = 1.0
alpha = 0.1

for step in range(20):
    prediction = np.dot(inputs, weights)
    error = (prediction - target) ** 2

    delta = prediction - target
    weight_deltas = delta * inputs

    weights = weights - alpha * weight_deltas

    new_prediction = np.dot(inputs, weights)
    new_error = (new_prediction - target) ** 2

    print(
        step,
        "weights =", weights,
        "prediction =", new_prediction,
        "error =", new_error
    )


0 weights = [0.48 0.19 0.11] prediction = 1.04 error = 0.001600000000000003
1 weights = [0.472 0.186 0.114] prediction = 1.016 error = 0.0002560000000000005
2 weights = [0.4688 0.1844 0.1156] prediction = 1.0064000000000002 error = 4.0960000000002345e-05
3 weights = [0.46752 0.18376 0.11624] prediction = 1.00256 error = 6.553599999999466e-06
4 weights = [0.467008 0.183504 0.116496] prediction = 1.0010240000000001 error = 1.0485760000002784e-06
5 weights = [0.4668032 0.1834016 0.1165984] prediction = 1.0004096 error = 1.6777216000000815e-07
6 weights = [0.46672128 0.18336064 0.11663936] prediction = 1.0001638400000001 error = 2.6843545600044962e-08
7 weights = [0.46668851 0.18334426 0.11665574] prediction = 1.000065536 error = 4.294967295989732e-09
8 weights = [0.4666754 0.1833377 0.1166623] prediction = 1.0000262144 error = 6.871947673630136e-10
9 weights = [0.46667016 0.18333508 0.11666492] prediction = 1.00001048576 error = 1.099511627790135e-10
10 weights = [0.46666806 0.18333403 0.

## 5.2 逐样本更新：从一个样本到多个样本

当有多个样本时，每个样本都要使用同一组 `weights`。这里先采用最直观的做法：一次取一个样本，算一次误差，立即更新。

`zip` 不是神经网络专用函数，它只是把多个可迭代对象按位置配对：

```python
list(zip(inputs, targets))
# [(第 1 个样本, 第 1 个目标), (第 2 个样本, 第 2 个目标), ...]
```

如果两个序列长度不同，`zip` 会在较短的序列结束时停止，所以训练数据和目标数组必须保持同样的样本数。

`epoch` 表示完整遍历一次所有样本。`zip(inputs, targets)` 会按照位置把样本和目标配对：第一个样本配第一个目标，第二个样本配第二个目标。

每个样本产生的更新都会作用于同一组权重，所以后面的样本会看到前面样本更新后的权重。`total_error` 越来越小，说明这一轮训练对整体数据的拟合变好。

这类方法也常被称为逐样本或在线更新。它的优点是每个样本都能及时提供反馈；代价是更新方向会受到样本顺序影响。

In [6]:
inputs = np.array([
    [2, 1, -1],
    [1, 0,  2],
])

targets = np.array([1.0, 0.5])
weights = np.array([0.5, 0.2, 0.1])
alpha = 0.1

for epoch in range(20):
    total_error = 0

    for sample, target in zip(inputs, targets):
        prediction = np.dot(sample, weights)
        delta = prediction - target

        weights = weights - alpha * delta * sample
        total_error += delta ** 2

    print(
        "epoch =", epoch,
        "weights =", weights,
        "total_error =", total_error
    )

epoch = 0 weights = [0.46 0.19 0.07] total_error = 0.049999999999999954
epoch = 1 weights = [0.442 0.186 0.054] total_error = 0.011599999999999997
epoch = 2 weights = [0.4338 0.1844 0.0456] total_error = 0.002756000000000005
epoch = 3 weights = [0.43002 0.18376 0.04124] total_error = 0.0006659600000000006
epoch = 4 weights = [0.428258 0.183504 0.038996] total_error = 0.0001628035999999995
epoch = 5 weights = [0.4274282 0.1834016 0.0378484] total_error = 4.011107600000139e-05
epoch = 6 weights = [0.42703378 0.18336064 0.03726436] total_error = 9.933397160000286e-06
epoch = 7 weights = [0.42684476 0.18334426 0.03696824] total_error = 2.4682497956001146e-06
epoch = 8 weights = [0.42675353 0.1833377  0.03681855] total_error = 6.146465297960938e-07
epoch = 9 weights = [0.42670922 0.18333508 0.03674304] total_error = 1.5327508539234568e-07
epoch = 10 weights = [0.4266876  0.18333403 0.03670503] total_error = 3.82569238190417e-08
epoch = 11 weights = [0.42667699 0.18333361 0.03668592] total_e

## 5.3 为什么两个样本还不能唯一确定权重？

当前数据只有 2 个样本，却有 3 个未知权重。数学上相当于：

```text
2 个方程 < 3 个未知数
```

因此仍然可能存在多组权重，都能把这两个样本拟合得很好。误差趋近于 0，只能说明当前样本被拟合了，不能说明模型已经找回唯一的真实参数。

更准确地说，是否能唯一确定参数取决于样本提供的独立约束，而不只是简单比较样本数和权重数。如果样本彼此重复，增加样本数量也可能没有增加新的信息。

增加第三个彼此独立的样本后，约束数量与权重数量相当，才有机会把权重确定下来。本 Notebook 后面的三样本实验会收敛到 `[0.5, 0.2, 0.1]`。

In [7]:
inputs = np.array([
    [2, 1, -1],
    [1, 0,  2],
    [0, 3,  1],
])

targets = np.array([1.1, 0.7, 0.7])
weights = np.array([0.1, 0.1, 0.1])
alpha = 0.1

for epoch in range(100):
    total_error = 0

    for sample, target in zip(inputs, targets):
        prediction = np.dot(sample, weights)
        delta = prediction - target

        weights = weights - alpha * delta * sample
        total_error += delta ** 2

    print(
        "epoch =", epoch,
        "weights =", weights,
        "total_error =", total_error
    )

epoch = 0 weights = [0.32  0.202 0.094] total_error = 0.9716
epoch = 1 weights = [0.4096  0.20456 0.08632] total_error = 0.17260544000000005
epoch = 2 weights = [0.453888  0.2039968 0.0880096] total_error = 0.04343634329600002
epoch = 3 weights = [0.47614464 0.2028407  0.09147789] total_error = 0.011581544359526396
epoch = 4 weights = [0.48750418 0.20184065 0.09447806] total_error = 0.003232633072658677
epoch = 5 weights = [0.49338396 0.20113343 0.09659972] total_error = 0.0009326955205868011
epoch = 6 weights = [0.4964653  0.20067637 0.0979709 ] total_error = 0.0002752296411612361
epoch = 7 weights = [0.49809737 0.20039535 0.09881395] total_error = 8.241477483160422e-05
epoch = 8 weights = [0.49896962 0.20022782 0.09931654] total_error = 2.4908032495161993e-05
epoch = 9 weights = [0.49943924 0.20012996 0.09961012] total_error = 7.571569560135526e-06
epoch = 10 weights = [0.49969363 0.2000736  0.09977921] total_error = 2.3098885054098447e-06
epoch = 11 weights = [0.4998321  0.20004145 

## 5.4 批量计算：为什么要使用 `inputs.T @ errors`？

逐样本更新之外，还可以先一次性计算所有样本的预测：

```python
predictions = inputs @ weights
errors = predictions - targets
```

形状关系是：

```text
inputs      (样本数, 特征数)
weights     (特征数,)
predictions (样本数,)
errors      (样本数,)
```

误差是按样本产生的，但我们需要的是每个特征对应的权重更新量。因此要把样本维度汇总到特征维度：

```python
weight_deltas = inputs.T @ errors
```

`inputs.T` 的形状是 `(特征数, 样本数)`，乘上 `(样本数,)` 的误差向量后，结果才是 `(特征数,)`，正好对应每个权重。第 `j` 个结果就是所有样本中第 `j` 个特征与误差的乘积之和。

注意：`inputs @ errors` 即使在方阵示例中可能不报错，语义也不对。它会按样本行汇总，而不是按特征列汇总。

In [4]:
inputs = np.array([
    [2, 1, -1],  # 样本 1
    [1, 0,  2],  # 样本 2
    [0, 3,  1],  # 样本 3
])

errors = np.array([0.1, -0.2, 0.05])

weight_deltas = inputs.T @ errors

print(weight_deltas)

[ 0.    0.25 -0.45]


## 5.5 批量梯度下降

批量梯度下降的流程是：

```text
所有样本预测
    ↓
得到所有误差
    ↓
按特征汇总误差
    ↓
统一更新一次权重
```

代码中的 `/ len(inputs)` 是取平均，避免样本数量变多时更新量仅仅因为样本多而变大。

逐样本更新和批量更新的核心目标相同，区别在于更新时机：前者每个样本更新一次，后者收集完一批样本的反馈后再更新一次。批量更新中的平均值 `/ len(inputs)` 让学习率的含义不随批大小简单放大。

本节可以用三个问题验收：

1. `predictions` 为什么是 `(样本数,)`？
2. `weight_deltas` 为什么是 `(特征数,)`？
3. 如果把 `inputs.T` 改成 `inputs`，哪一个维度的含义被弄反了？

In [8]:
import numpy as np

inputs = np.array([
    [2., 1., -1.],
    [1., 0.,  2.],
    [0., 3.,  1.],
])

targets = np.array([1.1, 0.7, 0.7])
weights = np.array([0.1, 0.1, 0.1])

alpha = 0.1

for epoch in range(200):
    # 所有样本同时预测
    predictions = inputs @ weights

    # 每个样本的误差
    errors = predictions - targets

    # 汇总到每个权重
    weight_deltas = (inputs.T @ errors) / len(inputs)

    # 一个 epoch 只更新一次
    weights = weights - alpha * weight_deltas

    total_error = np.mean(errors ** 2)

    if epoch % 10 == 0:
        print(
            "epoch =", epoch,
            "weights =", weights,
            "error =", total_error
        )

epoch = 0 weights = [0.17333333 0.16       0.10666667] error = 0.3533333333333333
epoch = 10 weights = [0.44021675 0.21971296 0.09424504] error = 0.007623901121085039
epoch = 20 weights = [0.48704508 0.2053329  0.0965697 ] error = 0.000377477894850494
epoch = 30 weights = [0.49706224 0.20132664 0.09891296] error = 2.0670476500592124e-05
epoch = 40 weights = [0.49931756 0.20032341 0.09970661] error = 1.167213842786028e-06
epoch = 50 weights = [0.49983934 0.2000781  0.09992566] error = 6.64878877952683e-08
epoch = 60 weights = [0.4999619  0.20001877 0.0999817 ] error = 3.79651983732305e-09
epoch = 70 weights = [0.49999093 0.2000045  0.09999556] error = 2.1692921526095236e-10
epoch = 80 weights = [0.49999784 0.20000108 0.09999893] error = 1.2397378349219546e-11
epoch = 90 weights = [0.49999948 0.20000026 0.09999974] error = 7.085385295201451e-13
epoch = 100 weights = [0.49999988 0.20000006 0.09999994] error = 4.0495155677070536e-14
epoch = 110 weights = [0.49999997 0.20000001 0.09999999] 

## 第 4–5 章小结

我们已经把训练过程从一个标量扩展到了向量和矩阵：

1. 一个输入、一个权重：理解误差和学习率；
2. 一个样本、多个权重：用点积得到预测，用向量更新权重；
3. 多个样本、多个权重：逐样本更新，或用矩阵乘法批量更新。

验收标准不是记住 `@` 或 `dot`，而是能解释每个维度代表什么，以及误差为什么必须汇总到权重对应的特征维度。

下一章将把网络扩展到多层：前向传播仍然使用点积，但误差需要通过链式法则传回前面的层，这就是反向传播。